# Deep Learning – Assignment 1
## Applying a Convolutional Neural Network to Blood Cell Image Classification

**TensorFlow / Keras** implementation adapted from the [official TensorFlow CNN tutorial](https://www.tensorflow.org/tutorials/images/cnn)

| | |
|---|---|
| **Name** | HOUT NGUONHENG |
| **Student ID** | 202624428 |
| **Date** | 2026 October 08 |

---

### Summary
This notebook builds a **four-class white blood cell classifier** (eosinophil, lymphocyte, monocyte, neutrophil).

A CNN from the TensorFlow tutorial is redesigned for **120×160** colour microscope images, trained with **Adam** and ReduceLROnPlateau, and evaluated on Kaggle TEST, BCCD originals, and LISC test.

**Colab run results (official numbers for the report, taken from `results_tf/`):**
- **Kaggle TEST: 87.21%** accuracy, macro F1 **0.876**
- BCCD originals (`dataset-master`, 347 images): **95.10%**, macro F1 0.959
- LISC test: **100.00%** (996 of 996; **not independent of TRAIN**)
- Best checkpoint: **epoch 24** (val loss 0.0016)

> **Report & code:** [GitHub – blood_cell_image_CNN_repo](https://github.com/nguonheng23dev/blood_cell_image_CNN_repo)

## How to run this notebook

### Google Colab
1. **Runtime → Change runtime type → GPU**
2. Run the **Colab setup** cell below (mount Drive)
3. Set `BASE` to your dataset folder on Drive
4. **Runtime → Run all** (or Shift+Enter cell by cell)

### Local / Anaconda
1. Skip the Colab mount cell
2. Set `BASE` to your local datasets folder
3. Run all cells top to bottom

### Expected dataset layout under `BASE`
```text
BASE/
├── dataset2-master/dataset2-master/images/TRAIN/{EOSINOPHIL,LYMPHOCYTE,MONOCYTE,NEUTROPHIL}/
├── dataset2-master/dataset2-master/images/TEST/{...}/
├── dataset-master/dataset-master/labels.csv
├── dataset-master/dataset-master/JPEGImages/
└── dataset2-master/LISC_dataset/test/{...}/
```

> **Note on reproducibility:** the seed is fixed (42) and op-determinism is requested, but GPU runs can still differ slightly between sessions. The numbers quoted in the markdown cells come from one specific run (saved in `results_tf/`); re-running may shift them by a small amount, so re-check them after a new run.

### Colab only – mount Google Drive
*Skip this cell if you run locally.*

In [ ]:
# ============ COLAB ONLY: Mount Drive ============
# Skip this cell on local / Anaconda
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except ImportError:
    print("Not in Colab — skip Drive mount.")

## Cell 0 – Setup & paths

Set `BASE` to the folder that contains `dataset2-master` and `dataset-master`.

In [ ]:
# ============ CELL 0: Setup & Determinism ============
import os, random, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
try:
    import matplotlib.pyplot as plt
except Exception:
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("Determinism notice:", e)

# ========== SET YOUR DATA PATH HERE ==========
# Colab example:
# BASE = Path("/content/drive/MyDrive/Blood_cell_image_CNN/datasets")
# Local example:
# BASE = Path(r"C:\Users\YourName\Blood_cell_image_CNN\datasets")
# BASE = Path("/Users/YourName/BloodCell_cell_image_CNN/datasets")

BASE = Path("/content/drive/MyDrive/Blood_cell_image_CNN/datasets")  # <-- CHANGE ME

KAGGLE = BASE / "dataset2-master" / "dataset2-master" / "images"
BCCD   = BASE / "dataset-master" / "dataset-master"
LISC   = BASE / "dataset2-master" / "LISC_dataset"
OUT    = Path("results_tf")
OUT.mkdir(exist_ok=True)

CLASSES  = ["EOSINOPHIL", "LYMPHOCYTE", "MONOCYTE", "NEUTROPHIL"]
IMG_SIZE = (120, 160)   # Height x Width (4:3)
BATCH    = 32
EPOCHS   = 25
LR       = 1e-3
EXT      = {".jpg", ".jpeg", ".png"}

print("TF version :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices("GPU"))
print("BASE exists:", BASE.exists(), "->", BASE)
print("KAGGLE     :", KAGGLE.exists())
print("BCCD       :", BCCD.exists())
print("LISC       :", LISC.exists())
print("Output dir :", OUT.resolve())

## Task 1 – Application and Dataset

### 1.1 Application
Microscope images of single **white blood cells (leukocytes)** are classified into one of four types.
The differential white-cell count is a common lab test; abnormal proportions indicate infection, allergy, parasitic disease or blood disorders. Manual counting is slow — automatic classification is a natural CNN application.

### 1.2 Dataset roles (from report)

| Dataset | Role | Size | Images |
|---------|------|------|--------|
| Kaggle augmented (TRAIN / TEST) | Train, val, internal test | 320×240 | TRAIN 9,957 · TEST 2,487 |
| Kaggle original (`dataset-master`) | Test only | 640×480 | 347 single-label |
| LISC test | Test only (not independent) | 224×224 | 996 |

### 1.3 Classes
| Class | Appearance | Cues |
|-------|------------|------|
| Eosinophil | Bi-lobed nucleus, orange-red granules | Colour & texture |
| Lymphocyte | Large round dark nucleus, thin cytoplasm | Round shape |
| Monocyte | Largest, kidney/horseshoe nucleus | Size & shape |
| Neutrophil | 3–5 connected lobes | Multi-lobed nucleus |

In [ ]:
# ============ TASK 1: Dataset listing ============
def files_from_folder(root):
    X, y = [], []
    if not Path(root).exists():
        print(f"WARNING: folder not found: {root}")
        return np.array([]), np.array([])
    for i, c in enumerate(CLASSES):
        folder = Path(root) / c
        if not folder.exists():
            print(f"WARNING: missing class folder: {folder}")
            continue
        for p in sorted(folder.glob("*")):
            if p.suffix.lower() in EXT:
                X.append(str(p))
                y.append(i)
    return np.array(X), np.array(y)

def files_from_bccd():
    labels_path = BCCD / "labels.csv"
    if not labels_path.exists():
        print(f"WARNING: labels.csv not found at {labels_path} — skipping BCCD originals")
        return np.array([]), np.array([])
    df = pd.read_csv(labels_path)
    X, y = [], []
    for _, r in df.iterrows():
        cat = str(r["Category"]).strip().upper()
        if cat in CLASSES:
            p = BCCD / "JPEGImages" / f"BloodImage_{int(r['Image']):05d}.jpg"
            if p.exists():
                X.append(str(p))
                y.append(CLASSES.index(cat))
    return np.array(X), np.array(y)

X_all,  y_all  = files_from_folder(KAGGLE / "TRAIN")
X_test, y_test = files_from_folder(KAGGLE / "TEST")
X_bccd, y_bccd = files_from_bccd()
X_lisc, y_lisc = files_from_folder(LISC / "test")

def counts(y):
    if len(y) == 0:
        return {}
    return {c: int((y == i).sum()) for i, c in enumerate(CLASSES)}

print("Kaggle TRAIN :", len(X_all),  counts(y_all))
print("Kaggle TEST  :", len(X_test), counts(y_test))
print("BCCD originals:", len(X_bccd), counts(y_bccd))
print("LISC test    :", len(X_lisc), counts(y_lisc))

In [ ]:
# Task 1 – Figure 1: original dataset screenshot
if len(X_all) == 0:
    raise SystemExit("No TRAIN images found. Fix BASE path in Cell 0 and re-run.")

fig, axes = plt.subplots(len(CLASSES), 4, figsize=(11, 9))
for i in range(len(CLASSES)):
    idxs = np.where(y_all == i)[0]
    chosen = np.random.choice(idxs, min(4, len(idxs)), replace=False)
    for j, k in enumerate(chosen):
        img = plt.imread(X_all[k])
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
        axes[i, j].set_title(f"{CLASSES[i]} {img.shape[1]}x{img.shape[0]}", fontsize=8)
plt.suptitle("Figure 1. Example images from each class (original size)", fontsize=12)
plt.tight_layout()
plt.savefig(OUT / "1_original_dataset.png", dpi=150)
plt.show()
print("Saved ->", OUT / "1_original_dataset.png")

## Task 2 – Data Preprocessing

| Step | What | Why |
|------|------|-----|
| **Resize** | 120 × 160 (H×W), keep 4:3 | Fixed input; 75% less compute |
| **Normalize** | ÷ 255 → [0, 1] | Stable gradients for Adam |
| **Labels** | One-hot vectors | Softmax + categorical CE |
| **Split** | 85% train / 15% val, stratified | Monitor convergence |
| **Augmentation** | Flip, rotate ±10%, zoom ±10%, contrast ±10% (train only) | Cells have no preferred orientation |
| **Extra tests** | BCCD + LISC same resize/normalize, never trained on | External-style checks |

In [ ]:
# ============ TASK 2: Preprocessing ============
X_tr, X_va, y_tr, y_va = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=SEED)

def load(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.one_hot(label, len(CLASSES))

def make_ds(X, y, shuffle=False):
    if len(X) == 0:
        return None
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED)
    return (ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(BATCH)
              .prefetch(tf.data.AUTOTUNE))

train_ds = make_ds(X_tr, y_tr, shuffle=True)
val_ds   = make_ds(X_va, y_va)
test_ds  = make_ds(X_test, y_test)
bccd_ds  = make_ds(X_bccd, y_bccd)
lisc_ds  = make_ds(X_lisc, y_lisc)

augment = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

print(f"Train: {len(X_tr):,} | Val: {len(X_va):,} | Test: {len(X_test):,}")
print(f"BCCD: {len(X_bccd):,} | LISC: {len(X_lisc):,}")

In [ ]:
# Figure 2 – Before / after preprocessing
fig, axes = plt.subplots(3, 4, figsize=(11, 8))
for j in range(min(4, len(X_tr))):
    p = X_tr[j]
    orig = plt.imread(p)
    proc, _ = load(p, y_tr[j])
    aug = augment(tf.expand_dims(proc, 0), training=True)[0]
    axes[0, j].imshow(orig)
    axes[0, j].set_title(f"Original {orig.shape[1]}x{orig.shape[0]}", fontsize=8)
    axes[1, j].imshow(proc)
    axes[1, j].set_title(f"Resized {IMG_SIZE[1]}x{IMG_SIZE[0]} /255", fontsize=8)
    axes[2, j].imshow(np.clip(aug.numpy() if hasattr(aug, 'numpy') else aug, 0, 1))
    axes[2, j].set_title("Augmented", fontsize=8)
    for i in range(3):
        axes[i, j].axis("off")
plt.suptitle("Figure 2. Row 1: original · Row 2: resized & normalised · Row 3: augmentation", fontsize=11)
plt.tight_layout()
plt.savefig(OUT / "2_preprocessing_before_after.png", dpi=150)
plt.show()
print("Saved ->", OUT / "2_preprocessing_before_after.png")

## Task 3 – Apply the TensorFlow CNN

### Changes from the CIFAR-10 tutorial (report §3.2)

| Item | Tutorial | This project | Reason |
|------|----------|--------------|--------|
| Input | 32×32×3 | **120×160×3** | Granules & lobes need more pixels |
| Output | 10 logits | **4 softmax** | Four cell types |
| Loss | Sparse CE | **Categorical CE** | One-hot labels |
| Conv | 3 layers (32,64,64) | **4 (32,64,128,128), same padding** | More depth |
| Pool | 2 max-pool | **4 max-pool** | 120×160 → 7×10 |
| Norm | none | **BatchNorm after each conv** | Stable training |
| Regularisation | none | **Dropout 0.3 + augmentation** | Dense layer overfits easily |
| Dense | 64 | **128** | More features |
| Optimizer | Adam | **Adam lr=0.001 + ReduceLROnPlateau, 25 epochs** | Task 4 |

In [ ]:
# ============ TASK 3: CNN Architecture ============
model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    augment,  # training only

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation="softmax"),
])

model.summary()
with open(OUT / "3_model_summary.txt", "w") as f:
    model.summary(print_fn=lambda s: f.write(s + "\n"))
print("Saved ->", OUT / "3_model_summary.txt")

### How the architecture works (report §3.5)
- **Conv + ReLU** – local patterns; non-linearity
- **Feature hierarchy** – edges → granules → nucleus lobes
- **Max pool** – shift tolerance, less compute
- **BatchNorm** – stable activations at lr=0.001
- **Dense(128) + Dropout(0.3)** – most parameters live here; dropout reduces memorisation
- **Softmax** – four class probabilities

Total ≈ **1.39 M** parameters (run output: 1,389,764; 1,389,060 trainable, 704 non-trainable).

## Task 4 – Optimization Technique

| Item | Choice |
|------|--------|
| Optimizer | **Adam** |
| Learning rate | **0.001**, halved by `ReduceLROnPlateau` (factor 0.5, patience 3, min 1e-5) |
| Batch size | 32 |
| Epochs | 25 |
| Loss | Categorical cross-entropy: \(L = -\log(p_k)\) |

**Why Adam:** momentum + per-weight scaling; robust default lr; handles noisy mini-batches well.

**Convergence checks:** training loss plateaus; validation loss reaches a minimum; accuracy curves stop trending; train–val gap stays reasonable.

**What happened in the Colab run (from `training_history_tf.csv`):** `ReduceLROnPlateau` fired three times — lr = 0.001 in epochs 1–7, 0.0005 in epochs 8–11, 0.00025 in epochs 12–16 and 0.000125 in epochs 17–25 (the CSV shows these rounded to 4 decimals as 0.0003 and 0.0001). The minimum lr (1e-5) was never reached.

In [ ]:
# ============ TASK 4: Compile + Train ============
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

ckpt = tf.keras.callbacks.ModelCheckpoint(
    str(OUT / "best_model.keras"),
    monitor="val_loss",
    save_best_only=True,
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[ckpt, lr_scheduler],
)

pd.DataFrame(history.history).round(4).to_csv(
    OUT / "training_history_tf.csv", index_label="epoch")
print("Training finished. History ->", OUT / "training_history_tf.csv")

## Task 5 – Training Results (accuracy & loss graphs)

All later evaluations use the **best checkpoint** (lowest validation loss, **epoch 24** in the Colab run).

| Quantity | Value |
|----------|-------|
| Training loss, first → last epoch | 1.3829 → 0.0137 (99.0 % decrease) |
| Validation loss, first → last epoch | 5.7949 → 0.0180 |
| Lowest validation loss | 0.0016 at epoch 24 |
| Train / validation accuracy at best epoch (24) | 99.61 % / 99.93 % |
| Train / validation accuracy at last epoch (25) | 99.60 % / 99.33 % |
| Validation-loss spikes (> 10 % jump) | 13 of 24 epoch-to-epoch changes |

In [ ]:
# ============ TASK 5: Graphs ============
h = history.history
ep = range(1, len(h["loss"]) + 1)
best_epoch = int(np.argmin(h["val_loss"])) + 1

# Combined figure (report Figure 5 style)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, h["accuracy"], label="train")
ax[0].plot(ep, h["val_accuracy"], label="validation")
ax[0].set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(ep, h["loss"], label="train")
ax[1].plot(ep, h["val_loss"], label="validation")
ax[1].axvline(best_epoch, ls="--", c="gray",
              label=f"lowest val loss (epoch {best_epoch})")
ax[1].set(xlabel="Epoch", ylabel="Loss", title="Loss")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle(f"Figure 5. Training curves (best epoch = {best_epoch})", fontsize=12)
plt.tight_layout()
plt.savefig(OUT / "5_training_curves.png", dpi=150)
plt.show()

# Separate graphs required by assignment Task 5
plt.figure(figsize=(6, 4))
plt.plot(ep, h["accuracy"], label="training")
plt.plot(ep, h["val_accuracy"], label="validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("Training vs validation accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT / "5a_accuracy.png", dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(ep, h["loss"], label="training")
plt.plot(ep, h["val_loss"], label="validation")
plt.axvline(best_epoch, ls="--", c="gray",
            label=f"lowest val loss (epoch {best_epoch})")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Training vs validation loss")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT / "5b_loss.png", dpi=150)
plt.show()

print(f"Best epoch: {best_epoch}")
print(f"Train loss first -> last: {h['loss'][0]:.4f} -> {h['loss'][-1]:.4f}")
print(f"Val   loss first -> last: {h['val_loss'][0]:.4f} -> {h['val_loss'][-1]:.4f}")

In [ ]:
# Convergence analysis (report §4.6)
def convergence_report(h):
    tl, vl = np.array(h["loss"]), np.array(h["val_loss"])
    ta, va = np.array(h["accuracy"]), np.array(h["val_accuracy"])
    n = len(tl)
    k = min(5, n)
    best = int(vl.argmin()) + 1
    first_close = int(np.argmax(vl <= vl.min() * 1.02)) + 1   # first epoch within 2% of best
    rises = int(np.sum(np.diff(vl) > 0.10 * np.maximum(vl[:-1], 1e-8)))
    lines = [
        f"Epochs run                          : {n}",
        f"1) Train loss  first -> last        : {tl[0]:.4f} -> {tl[-1]:.4f}  "
        f"({100*(1-tl[-1]/max(tl[0],1e-8)):.1f}% decrease)",
        f"   Val loss    first -> last        : {vl[0]:.4f} -> {vl[-1]:.4f}",
        f"2) Train loss change, last {k} epochs : {abs(tl[-1]-tl[-k]):.4f}",
        f"   Val loss std, last {k} epochs     : {vl[-k:].std():.4f}",
        f"3) Lowest val loss                  : {vl.min():.4f} at epoch {best}",
        f"4) First epoch within 2% of best    : Epoch {first_close}",
        f"5) Val-loss spikes (>10% jump)      : {rises}",
        f"   Train/val accuracy at the end    : {ta[-1]:.4f} / {va[-1]:.4f}",
    ]
    return "\n".join(lines)

report = convergence_report(history.history)
print(report)
(OUT / "4_convergence_analysis.txt").write_text(report)
print("Saved ->", OUT / "4_convergence_analysis.txt")

## Task 6 – Effect of CNN on the Dataset

Evaluate the best checkpoint (**epoch 24** in the Colab run).

**Colab run results:**
| Set | Accuracy | Macro F1 |
|-----|----------|----------|
| Kaggle TEST (2,487) | **87.21%** | **0.876** |
| BCCD originals (347) | 95.10% | 0.959 |
| LISC test (996) | 100.00% | 1.000 |

**Per-class results on Kaggle TEST:**
| Class | Precision | Recall | F1 |
|-------|-----------|--------|----|
| Eosinophil | 0.940 | 0.830 | 0.882 |
| Lymphocyte | 1.000 | 0.958 | 0.979 |
| Monocyte | 0.975 | 0.750 | 0.848 |
| Neutrophil | 0.685 | 0.950 | 0.796 |

- **Lymphocyte** easiest (F1 0.979); **neutrophil** high recall / low precision (over-predicted — roughly 270 false positives, estimated from the rounded precision/recall because no confusion matrix was saved).
- Kaggle TRAIN/TEST are augmented — test can be optimistic.
- BCCD originals are probably the source of the augmented images — also optimistic; only 20 monocytes and 33 lymphocytes, so those per-class scores are unreliable.
- LISC names match Kaggle TRAIN → **not independent** (the 100% mostly measures recognition of images already seen).
- Validation (99.93% at the best epoch) is about 12.7 points above Kaggle TEST: validation images come from the same augmented pool as TRAIN.
- Primary number for the report: **Kaggle TEST 87.21%**.

In [ ]:
# ============ TASK 6: Evaluation ============
model = tf.keras.models.load_model(OUT / "best_model.keras")

def evaluate(ds, y_true, name):
    if ds is None or len(y_true) == 0:
        print(f"\n=== {name}: skipped (no data) ===")
        return None
    prob = model.predict(ds, verbose=0)
    pred = prob.argmax(1)
    acc = float((pred == y_true).mean())
    rep = classification_report(
        y_true, pred,
        labels=range(len(CLASSES)),
        target_names=CLASSES,
        digits=3,
        zero_division=0,
    )
    out_path = OUT / f"6_report_{name.replace(' ', '_')}.txt"
    with open(out_path, "w") as f:
        f.write(f"accuracy {acc:.4f}\n{rep}")
    print(f"\n=== {name}  accuracy = {acc:.4f} ===")
    print(rep)
    return acc

# Names match the files saved in results_tf/ (6_report_Kaggle_test.txt, 6_report_BCCD_external.txt, 6_report_LISC_test.txt)
results = {
    "Kaggle test (internal)": evaluate(test_ds, y_test, "Kaggle test"),
    "Kaggle originals (dataset-master)": evaluate(bccd_ds, y_bccd, "BCCD external"),
    "LISC_dataset test": evaluate(lisc_ds, y_lisc, "LISC test"),
}
json.dump({k: v for k, v in results.items() if v is not None},
          open(OUT / "6_summary.json", "w"), indent=2)
print("\nSummary:", results)
print("Saved ->", OUT / "6_summary.json")

### Grad-CAM – what patterns did the CNN learn? (report §6.2)
Heatmaps on the last conv layer (red = strongest influence). Qualitative sample only (two random Kaggle TEST images per class).

In the Colab run **6 of the 8 images were classified correctly**; the two errors (an eosinophil and a monocyte) were both predicted as neutrophil, the same direction as the full-test-set errors. Both lymphocytes and the correctly classified monocyte have a clear hot spot on the white cell itself. For the wrong eosinophil the hot spot falls on red blood cells instead, and for the first eosinophil and both neutrophils the heatmap is diffuse with no clear hot spot. Eight images cannot show this reliably — treat it as a hypothesis, not a finding.

In [ ]:
# ============ TASK 6: Grad-CAM ============
try:
    conv_layers = [l for l in model.layers if isinstance(l, layers.Conv2D)]
    last_conv = conv_layers[-1]

    def grad_cam(x):
        h_ = x[None, ...]
        with tf.GradientTape() as tape:
            for l in model.layers:
                if l.name == "augmentation":
                    continue
                h_ = l(h_, training=False)
                if l is last_conv:
                    conv_out = h_
                    tape.watch(conv_out)
            preds = h_
            cls = tf.argmax(preds[0])
            score = preds[:, cls]
        grads = tape.gradient(score, conv_out)
        w = tf.reduce_mean(grads, axis=(1, 2))
        cam = tf.reduce_sum(conv_out * w[:, None, None, :], axis=-1)[0]
        cam = tf.nn.relu(cam)
        cam = cam / (tf.reduce_max(cam) + 1e-8)
        cam = tf.image.resize(cam[None, ..., None], IMG_SIZE)[0, ..., 0].numpy()
        return cam, int(cls)

    fig, axes = plt.subplots(len(CLASSES), 4, figsize=(11, 9))
    for i, c in enumerate(CLASSES):
        idxs = np.where(y_test == i)[0]
        if len(idxs) == 0:
            continue
        chosen = np.random.choice(idxs, min(2, len(idxs)), replace=False)
        for j, k in enumerate(chosen):
            x, _ = load(X_test[k], y_test[k])
            cam, pred = grad_cam(x)
            axes[i, 2*j].imshow(x)
            axes[i, 2*j].axis("off")
            axes[i, 2*j].set_title(c, fontsize=8)
            axes[i, 2*j+1].imshow(x)
            axes[i, 2*j+1].imshow(cam, cmap="jet", alpha=0.45)
            axes[i, 2*j+1].axis("off")
            axes[i, 2*j+1].set_title(f"Grad-CAM -> {CLASSES[pred]}", fontsize=8)
    plt.suptitle("Figure 6. Grad-CAM on two Kaggle-test images per class", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / "6_gradcam.png", dpi=150)
    plt.show()
    print("Saved ->", OUT / "6_gradcam.png")
except Exception as e:
    print("Grad-CAM skipped:", e)

## Task 7 – Future Extension: Transfer Learning (EfficientNet-B0)

| Architecture | ~Params | Why considered |
|--------------|---------|----------------|
| **EfficientNet-B0** | **5.3 M** | Best accuracy per parameter; fits Colab GPU; 224×224 input close to cell sizes |
| ResNet50 | 25 M | Reliable but heavier |
| MobileNetV2 | 3.5 M | Lighter, slightly lower accuracy |

**How transfer learning would help**
1. **Feature extraction** – freeze ImageNet weights; train new GlobalAvgPool → Dropout → Dense(4)
2. **Fine-tuning** – unfreeze last blocks with very small lr (~1e-5)

Expected: higher accuracy on hard classes, fewer epochs, less overfitting.  
*(Not implemented in this assignment — future work only.)*

## Overall conclusion (Colab run)

- Tutorial CNN adapted: 120×160, 4 conv blocks, BatchNorm, Dropout 0.3, 4-way softmax (~1.39 M params).
- Adam (lr=0.001 + ReduceLROnPlateau, three reductions), **best checkpoint epoch 24**.
- **Kaggle TEST: 87.21%** (macro F1 0.876) — primary result.
- BCCD originals: 95.10% (macro F1 0.959); LISC test: 100.00% (overlaps TRAIN — not independent).
- Lymphocyte easiest (F1 0.979); neutrophil over-predicted (precision 0.685, recall 0.950).
- Validation was unstable for the first ~16 epochs and calmer afterwards; validation accuracy (99.93%) is about 12.7 points above test accuracy, so the validation score is optimistic.
- Future work: EfficientNet-B0 transfer learning.

---
Outputs → `results_tf/`.  
Repo: https://github.com/nguonheng23dev/blood_cell_image_CNN_repo

In [ ]:
# ============ Finish: list outputs ============
print("Notebook completed.")
print("Outputs in:", OUT.resolve())
for p in sorted(OUT.glob("*")):
    print(" ", p.name, f"({p.stat().st_size // 1024} KB)" if p.is_file() else "")

### Colab: download results

In [ ]:
# ============ COLAB: zip & download results ============
try:
    from google.colab import files
    import shutil
    zip_path = Path("/content/results_tf_bundle")
    shutil.make_archive(str(zip_path), "zip", root_dir=str(OUT))
    files.download(str(zip_path) + ".zip")
    print("Download started: results_tf_bundle.zip")
except ImportError:
    print("Not in Colab — copy the results_tf/ folder manually.")
except Exception as e:
    print("Download helper:", e)